# Controlled Leakage Injection + Paper Outputs

Notebook này tập trung vào phần **chứng minh leakage handling** (thuyết phục nhất) và gói output cho paper:

1) **Controlled leakage injection**  
   - Thêm 1–2 feature leak giả (`leak_direct = y`, `leak_hash = f(y)`)  
   - Đo: method nào drop đúng leak, và mức *false-drop*.

2) **Paper-ready artifacts**
   - Bảng tổng hợp method × metrics + #features (đọc từ `results.csv` nếu có)
   - Hình SHAP **trước/sau cleaning** + top features bị drop
   - Hình sensitivity τ (đọc từ `sensitivity_tau.csv` nếu có)
   - Bảng/hình detection rate (injection)

> Protocol: không dùng test cho feature selection. `test` chỉ để chấm điểm cuối.

In [ ]:
# !pip -q install shap lightgbm scikit-learn
import os
import time
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import f1_score, recall_score, roc_auc_score, confusion_matrix

In [ ]:
# =========================
# CONFIG (align with Notebook 01)
# =========================
CSV_PATH    = "/kaggle/input/datasets/tranvannha/train-test-network-2/train_test_network.csv"
TARGET_COL  = "label"

TEST_SIZE   = 0.2
VAL_FS_SIZE = 0.2
SEEDS       = [42, 43, 44, 45, 46]

MANUAL_IP_PORT = ["src_ip", "dst_ip", "src_port", "dst_port"]
GPU_CARDINALITY_LIMIT = 50

TAU_DEFAULT = 4.0
SHAP_SAMPLE_VAL = 10000
PERM_N_REPEATS  = 5

USE_GPU = False
LGBM_FIXED = dict(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=0,
    n_jobs=-1,
    objective="binary",
    verbose=-1
)

METHODS = [
    ("baseline0_manual_ip_port", dict(tau=TAU_DEFAULT)),
    ("perm_importance",          dict(tau=TAU_DEFAULT, n_repeats=PERM_N_REPEATS)),
    ("mutual_info",              dict(tau=TAU_DEFAULT, mi_eps=1e-6)),
    ("logreg_l1",                dict(C=0.2)),
    ("tree_based",               dict(tau=TAU_DEFAULT, top_k=None)),
    ("shap_method",              dict(tau=TAU_DEFAULT, shap_sample=SHAP_SAMPLE_VAL)),
]

print("Config OK.")

In [ ]:
# ---------- shared helpers (subset from Notebook 01) ----------
def load_raw_xy(csv_path: str, target_col: str):
    data = pd.read_csv(csv_path)
    X = data.drop(columns=[target_col])
    y = data[target_col].astype(int)
    return X, y

def preprocess_train_test(X_train_raw, X_test_raw, manual_drop, gpu_cardinality_limit=50):
    drop_cols = [c for c in manual_drop if c in X_train_raw.columns]
    X_train = X_train_raw.drop(columns=drop_cols).copy()
    X_test  = X_test_raw.drop(columns=drop_cols).copy()

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in numeric_cols]

    if len(numeric_cols) > 0:
        imp = SimpleImputer(strategy="median")
        X_train[numeric_cols] = imp.fit_transform(X_train[numeric_cols])
        X_test[numeric_cols]  = imp.transform(X_test[numeric_cols])

    if len(cat_cols) > 0:
        X_train[cat_cols] = X_train[cat_cols].astype(str).fillna("unknown")
        X_test[cat_cols]  = X_test[cat_cols].astype(str).fillna("unknown")

        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols])
        X_test[cat_cols]  = enc.transform(X_test[cat_cols])

    safe_cat = []
    for c in cat_cols:
        nun = int(pd.Series(X_train[c]).nunique())
        if nun <= gpu_cardinality_limit:
            X_train[c] = X_train[c].astype(int).astype("category")
            X_test[c]  = X_test[c].astype(int).astype("category")
            safe_cat.append(c)
        else:
            X_train[c] = X_train[c].astype(int)
            X_test[c]  = X_test[c].astype(int)

    return X_train, X_test, safe_cat, drop_cols

def compute_scale_pos_weight(y_train):
    neg = int((y_train == 0).sum())
    pos = int((y_train == 1).sum())
    return (neg / pos) if pos > 0 else 1.0

def build_lgbm_classifier(seed, scale_pos_weight):
    params = dict(LGBM_FIXED)
    params["random_state"] = seed
    params["scale_pos_weight"] = float(scale_pos_weight)
    if USE_GPU:
        params["device_type"] = "gpu"
        params["n_jobs"] = 1
    return lgb.LGBMClassifier(**params)

def dominance_drop(scores: pd.Series, tau: float):
    if scores is None or len(scores) < 2:
        return []
    s = scores.sort_values(ascending=False)
    if s.iloc[1] <= 0:
        return []
    if (float(s.iloc[0]) / float(s.iloc[1])) > float(tau):
        return [str(s.index[0])]
    return []

def tune_threshold_max_f1(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 19)
    f1s = []
    for th in thresholds:
        f1s.append(f1_score(y_true, (y_prob >= th).astype(int), pos_label=1, zero_division=0))
    return float(thresholds[int(np.argmax(f1s))])

def train_and_eval(X_tr, y_tr, X_val, y_val, X_te, y_te, safe_cat, seed):
    spw = compute_scale_pos_weight(y_tr)
    model = build_lgbm_classifier(seed, spw)
    try:
            model.fit(X_tr, y_tr, categorical_feature=safe_cat, eval_set=[(X_val, y_val)], eval_metric="auc", verbose=False)
    except TypeError:
            model.fit(X_tr, y_tr, categorical_feature=safe_cat, eval_set=[(X_val, y_val)], eval_metric="auc", )

    p_val = model.predict_proba(X_val)[:, 1]
    p_te  = model.predict_proba(X_te)[:, 1]
    th = tune_threshold_max_f1(y_val.values, p_val)
    yhat = (p_te >= th).astype(int)

    f1 = f1_score(y_te.values, yhat, pos_label=1, zero_division=0)
    rec = recall_score(y_te.values, yhat, pos_label=1, zero_division=0)
    auc_ = roc_auc_score(y_te.values, p_te)
    tn, fp, fn, tp = confusion_matrix(y_te.values, yhat).ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return dict(f1_pos=float(f1), recall_pos=float(rec), auc=float(auc_), fnr=float(fnr), best_th=float(th))

print("Helpers ready.")

## 1) Controlled leakage injection benchmark

Injection features:
- `leak_direct = y`
- `leak_hash = y * 2654435761 (bit-mix)`

Kết quả:
- detection rate: % runs drop đúng leak
- false-drop: số feature bị drop (không tính leak features)

In [ ]:
def inject_leak_features(X: pd.DataFrame, y: pd.Series):
    X2 = X.copy()
    X2["leak_direct"] = y.values.astype(int)
    X2["leak_hash"]   = (y.values.astype(np.uint32) * np.uint32(2654435761)).astype(np.uint32).astype(np.int64)
    return X2

def get_drop_list(method, X_train_fs, y_train_fs, X_val_fs, y_val_fs, safe_cat, params):
    tau = float(params.get("tau", TAU_DEFAULT))

    if method == "baseline0_manual_ip_port":
        return [], {}

    if method in ["perm_importance", "shap_method"]:
        spw = compute_scale_pos_weight(y_train_fs)
        model = build_lgbm_classifier(seed=0, scale_pos_weight=spw)
        model.fit(X_train_fs, y_train_fs, categorical_feature=safe_cat)

    if method == "perm_importance":
        pi = permutation_importance(
            model, X_val_fs, y_val_fs,
            scoring="roc_auc",
            n_repeats=int(params.get("n_repeats", PERM_N_REPEATS)),
            random_state=0,
            n_jobs=-1
        )
        scores = pd.Series(pi.importances_mean, index=X_val_fs.columns).sort_values(ascending=False)
        drop = scores[scores <= 0].index.tolist()
        drop += dominance_drop(scores, tau=tau)
        return sorted(set(drop)), {"scores": scores}

    if method == "mutual_info":
        mi_eps = float(params.get("mi_eps", 1e-6))
        discrete_mask = np.array([str(X_train_fs[c].dtype) == "category" for c in X_train_fs.columns], dtype=bool)

        X_num = X_train_fs.copy()
        for c in X_num.columns:
            if str(X_num[c].dtype) == "category":
                X_num[c] = X_num[c].cat.codes.astype(int)

        mi = mutual_info_classif(
            X_num.values, y_train_fs.values,
            discrete_features=discrete_mask,
            random_state=0
        )
        scores = pd.Series(mi, index=X_train_fs.columns).sort_values(ascending=False)
        drop = scores[scores <= mi_eps].index.tolist()
        drop += dominance_drop(scores, tau=tau)
        return sorted(set(drop)), {"scores": scores}

    if method == "logreg_l1":
        C = float(params.get("C", 0.2))
        X_tr = X_train_fs.copy()
        for c in X_tr.columns:
            if str(X_tr[c].dtype) == "category":
                X_tr[c] = X_tr[c].cat.codes.astype(int)
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr.values)
        clf = LogisticRegression(penalty="l1", solver="saga", C=C, max_iter=5000, n_jobs=-1, random_state=0)
        clf.fit(X_tr_s, y_train_fs.values)
        coef = np.abs(clf.coef_).ravel()
        scores = pd.Series(coef, index=X_train_fs.columns).sort_values(ascending=False)
        drop = scores[scores <= 1e-12].index.tolist()
        return sorted(set(drop)), {"scores": scores}

    if method == "tree_based":
        top_k = params.get("top_k", None)
        X_tr = X_train_fs.copy()
        for c in X_tr.columns:
            if str(X_tr[c].dtype) == "category":
                X_tr[c] = X_tr[c].cat.codes.astype(int)
        clf = ExtraTreesClassifier(n_estimators=400, random_state=0, n_jobs=-1, class_weight="balanced_subsample")
        clf.fit(X_tr.values, y_train_fs.values)
        imp = pd.Series(clf.feature_importances_, index=X_train_fs.columns).sort_values(ascending=False)
        drop = imp[imp <= 0].index.tolist()
        if top_k is not None:
            keep = set(imp.head(int(top_k)).index.tolist())
            drop = sorted(set(X_train_fs.columns) - keep)
        drop += dominance_drop(imp, tau=tau)
        return sorted(set(drop)), {"scores": imp}

    if method == "shap_method":
        shap_sample = int(params.get("shap_sample", SHAP_SAMPLE_VAL))
        X_val_sample = X_val_fs.sample(n=min(shap_sample, len(X_val_fs)), random_state=0)
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_val_sample)
        sv = sv[1] if isinstance(sv, list) else sv
        mean_abs = np.abs(sv).mean(axis=0)
        scores = pd.Series(mean_abs, index=X_val_sample.columns).sort_values(ascending=False)
        drop = scores[scores <= 0].index.tolist()
        drop += dominance_drop(scores, tau=tau)
        return sorted(set(drop)), {"scores": scores}

    raise ValueError(method)

def run_injection_one_seed(seed: int):
    X_raw, y = load_raw_xy(CSV_PATH, TARGET_COL)
    X_raw = inject_leak_features(X_raw, y)

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=TEST_SIZE, random_state=seed, stratify=y
    )

    # Preprocess (manual IP/port)
    X_train_p, X_test_p, safe_cat, _ = preprocess_train_test(
        X_train_raw, X_test_raw, manual_drop=MANUAL_IP_PORT, gpu_cardinality_limit=GPU_CARDINALITY_LIMIT
    )

    # FS split
    X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
        X_train_p, y_train, test_size=VAL_FS_SIZE, random_state=seed, stratify=y_train
    )

    rows = []
    for method, params in METHODS:
        drop_cols, _ = get_drop_list(method, X_train_fs, y_train_fs, X_val_fs, y_val_fs, safe_cat, params)
        leak_direct_hit = ("leak_direct" in drop_cols)
        leak_hash_hit   = ("leak_hash" in drop_cols)

        # false-drop excluding injected leaks
        drop_wo_leaks = [c for c in drop_cols if c not in ["leak_direct","leak_hash"]]
        rows.append({
            "seed": seed,
            "method": method,
            "drop_leak_direct": int(leak_direct_hit),
            "drop_leak_hash": int(leak_hash_hit),
            "drop_both": int(leak_direct_hit and leak_hash_hit),
            "false_drop_count": int(len(drop_wo_leaks)),
            "n_dropped_total": int(len(drop_cols)),
        })
    return pd.DataFrame(rows)

inj_df = pd.concat([run_injection_one_seed(s) for s in SEEDS], ignore_index=True)
inj_df

In [ ]:
# Detection rate summary
det_summary = inj_df.groupby("method")[["drop_leak_direct","drop_leak_hash","drop_both","false_drop_count","n_dropped_total"]].agg(["mean","std"]).reset_index()
det_summary.columns = ["_".join([c for c in col if c]) for col in det_summary.columns.values]
det_summary = det_summary.rename(columns={"method_": "method"})
det_summary

In [ ]:
# Plot: detection rate (drop_both) per method
plt.figure(figsize=(8,4))
x = det_summary["method"]
y = det_summary["drop_both_mean"]
plt.bar(x, y)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Detection rate (drop both leaks)")
plt.title("Controlled Injection: Detection Rate")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig("fig_injection_detection_rate.png", dpi=200)
plt.show()

print("Saved figure: fig_injection_detection_rate.png")

## 2) Paper table: method × metrics + #features

- Nếu đã chạy Notebook 01, notebook này sẽ đọc `results.csv` và tạo:
  - `paper_table_main.csv`
  - `paper_table_main.tex` (optional)

In [ ]:
if os.path.exists("results.csv"):
    results = pd.read_csv("/kaggle/input/13-02-2026-baseline/results/results.csv")
    metric_cols = ["f1_pos", "recall_pos", "fnr", "auc", "n_features_kept"]
    summary = results.groupby("method")[metric_cols].agg(["mean","std"]).reset_index()
    # flatten columns
    summary.columns = ["_".join([c for c in col if c]) for col in summary.columns.values]
    summary = summary.rename(columns={"method_":"method"})
    summary
else:
    print("results.csv not found. Please run Notebook 01 first (or copy results.csv here).")

In [ ]:
if "summary" in globals():
    # Create pretty mean±std strings for paper table
    def pm(m, s, digits=4):
        return f"{m:.{digits}f}±{s:.{digits}f}"

    paper = pd.DataFrame({
        "method": summary["method"],
        "F1_pos": [pm(m,s,4) for m,s in zip(summary["f1_pos_mean"], summary["f1_pos_std"])],
        "Recall_pos": [pm(m,s,4) for m,s in zip(summary["recall_pos_mean"], summary["recall_pos_std"])],
        "FNR": [pm(m,s,4) for m,s in zip(summary["fnr_mean"], summary["fnr_std"])],
        "AUC": [pm(m,s,4) for m,s in zip(summary["auc_mean"], summary["auc_std"])],
        "#features_kept": [pm(m,s,1) for m,s in zip(summary["n_features_kept_mean"], summary["n_features_kept_std"])],
    })
    paper

In [ ]:
if "paper" in globals():
    paper.to_csv("paper_table_main.csv", index=False)
    print("Saved: paper_table_main.csv")

    # Optional: LaTeX table string (copy-paste into paper)
    latex = paper.to_latex(index=False)
    with open("paper_table_main.tex", "w") as f:
        f.write(latex)
    print("Saved: paper_table_main.tex")

## 3) SHAP figure before/after cleaning + top dropped features

Quy trình:
- Chọn 1 seed (mặc định 42)
- Train baseline model trên `train_fs` (không drop thêm)
- Tính SHAP mean|SHAP| trên `val_fs` (**before cleaning**) và lưu hình
- Áp dụng SHAP method (dominance + SHAP=0) để drop
- Retrain trên `train_fs_clean`, tính SHAP trên `val_fs_clean` (**after cleaning**) và lưu hình
- Trích top features bị drop (theo SHAP before) và lưu bar chart

In [ ]:
PAPER_SEED = 42

def shap_summary_plot_to_file(shap_vals, X, out_path, title):
    plt.figure()
    shap.summary_plot(shap_vals, X, show=False, plot_size=(10,6))
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()

# Load non-injected data for figures
X_raw, y = load_raw_xy(CSV_PATH, TARGET_COL)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=TEST_SIZE, random_state=PAPER_SEED, stratify=y
)
X_train_p, X_test_p, safe_cat, _ = preprocess_train_test(
    X_train_raw, X_test_raw, manual_drop=MANUAL_IP_PORT, gpu_cardinality_limit=GPU_CARDINALITY_LIMIT
)
X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
    X_train_p, y_train, test_size=VAL_FS_SIZE, random_state=PAPER_SEED, stratify=y_train
)

# ---- BEFORE cleaning ----
spw = compute_scale_pos_weight(y_train_fs)
model_before = build_lgbm_classifier(seed=PAPER_SEED, scale_pos_weight=spw)
model_before.fit(X_train_fs, y_train_fs, categorical_feature=safe_cat)

X_val_sample = X_val_fs.sample(n=min(SHAP_SAMPLE_VAL, len(X_val_fs)), random_state=0)
expl_before = shap.TreeExplainer(model_before)
sv_before = expl_before.shap_values(X_val_sample)
sv_before = sv_before[1] if isinstance(sv_before, list) else sv_before

shap_summary_plot_to_file(sv_before, X_val_sample, "fig_shap_before.png", "SHAP summary (before cleaning)")

mean_abs_before = pd.Series(np.abs(sv_before).mean(axis=0), index=X_val_sample.columns).sort_values(ascending=False)

# ---- SHAP method drop list ----
drop_cols = mean_abs_before[mean_abs_before <= 0].index.tolist()
if len(mean_abs_before) >= 2 and mean_abs_before.iloc[1] > 0 and (mean_abs_before.iloc[0]/mean_abs_before.iloc[1]) > TAU_DEFAULT:
    drop_cols.append(mean_abs_before.index[0])
drop_cols = sorted(set(drop_cols))

print("Dropped by SHAP method:", len(drop_cols))
print("Top 10 dropped (by SHAP before):")
print([c for c in mean_abs_before.index if c in drop_cols][:10])

# ---- AFTER cleaning ----
X_train_c = X_train_fs.drop(columns=drop_cols)
X_val_c   = X_val_fs.drop(columns=drop_cols)
safe_cat_c = [c for c in safe_cat if c in X_train_c.columns]

model_after = build_lgbm_classifier(seed=PAPER_SEED, scale_pos_weight=spw)
model_after.fit(X_train_c, y_train_fs, categorical_feature=safe_cat_c)

X_val_c_sample = X_val_c.sample(n=min(SHAP_SAMPLE_VAL, len(X_val_c)), random_state=0)
expl_after = shap.TreeExplainer(model_after)
sv_after = expl_after.shap_values(X_val_c_sample)
sv_after = sv_after[1] if isinstance(sv_after, list) else sv_after

shap_summary_plot_to_file(sv_after, X_val_c_sample, "fig_shap_after.png", "SHAP summary (after cleaning)")

print("Saved: fig_shap_before.png, fig_shap_after.png")

In [ ]:
# Top dropped features bar chart (using SHAP before)
dropped_ranked = mean_abs_before.loc[[c for c in mean_abs_before.index if c in drop_cols]]
topk = dropped_ranked.head(20)

plt.figure(figsize=(8,6))
plt.barh(topk.index[::-1], topk.values[::-1])
plt.xlabel("mean |SHAP| (before)")
plt.title("Top dropped features by SHAP method (before cleaning)")
plt.tight_layout()
plt.savefig("fig_top_dropped_features.png", dpi=220, bbox_inches="tight")
plt.show()

print("Saved: fig_top_dropped_features.png")

## 4) Sensitivity τ figure (optional)

Nếu có `sensitivity_tau.csv` (từ Notebook 02), notebook này sẽ render lại và lưu `fig_sensitivity_tau.png`.

In [ ]:
if os.path.exists("sensitivity_tau.csv"):
    sens = pd.read_csv("/kaggle/input/13-02-2026-baseline/results/sensitivity_tau.csv").sort_values("tau")
    plt.figure(figsize=(8,5))
    plt.plot(sens["tau"], sens["f1_pos"], marker="o", label="F1_pos")
    plt.plot(sens["tau"], sens["recall_pos"], marker="o", label="Recall_pos")
    plt.xscale("log")
    plt.xlabel("tau (log-scale)")
    plt.ylabel("metric")
    plt.title("Sensitivity: Metrics vs tau")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig("fig_sensitivity_tau.png", dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved: fig_sensitivity_tau.png")
else:
    print("sensitivity_tau.csv not found (run Notebook 02 first).")